# Tabular Safe RL Baselines

This notebook compares MASA's tabular safe RL baselines on the same constrained `colour_grid_world` setup. The runs are intentionally tiny: the goal is to understand the safety mechanism each algorithm adds, not to produce benchmark rankings.

The matching static docs page is at `docs/Tutorials/Algorithms/Tabular Safe RL Baselines.md`.

## CPU-first setup

Set CPU-oriented environment variables before importing MASA/JAX modules (for portability).

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

from pprint import pprint

ACTION_NAMES = {0: "left", 1: "right", 2: "down", 3: "up", 4: "stay"}

## Imports

The five classes below correspond to the registered algorithm names `q_learning`, `q_learning_lambda`, `lcrl`, `sem`, and `recreg`.

In [ ]:
from IPython.display import Markdown, display

from masa.plugins.helpers import load_plugins
from masa.common.utils import make_env
from masa.envs.tabular.colour_grid_world import BLUE_STATE, GOAL_STATE, cost_fn, label_fn
from masa.algorithms.tabular import LCRL, QL, QL_Lambda, RECREG, SEM
from notebooks.tutorials.helpers.tabular_safe_rl_baselines import (
    algorithm_state_summary,
    markdown_table,
    smoke_run_note,
)

load_plugins()

## Shared environment

All algorithms see the same CMDP environment. The task reward is sparse and positive at `goal`; safety cost is positive at the blue state. This is a small enough setting for notebook execution but still separates reward and safety.

In [ ]:
ENV_ID = "colour_grid_world"
CONSTRAINT = "cmdp"
MAX_EPISODE_STEPS = 40
BUDGET = 0.0


def make_train_env():
    return make_env(
        ENV_ID,
        CONSTRAINT,
        MAX_EPISODE_STEPS,
        label_fn=label_fn,
        cost_fn=cost_fn,
        budget=BUDGET,
    )


def make_eval_env():
    return make_env(
        "colour_grid_world",
        "cmdp",
        40,
        label_fn=label_fn,
        cost_fn=cost_fn,
        budget=0.0,
    )

print("unsafe blue state:", BLUE_STATE)
print("rewarding goal state:", GOAL_STATE)

## Probe the reward/safety signals first

Before training, run two deterministic scripts. The unsafe script reaches `blue`; the goal script reaches `goal` without visiting `blue`. This gives the metrics that every algorithm will later receive through `info`.

In [ ]:
def run_script(seed, actions):
    env = make_train_env()
    obs, info = env.reset(seed=seed)
    rows = [
        {
            "event": "reset",
            "obs": int(obs),
            "labels": sorted(info["labels"]),
            "constraint_step": info["constraint"]["step"],
        }
    ]
    for step, action in enumerate(actions, start=1):
        obs, reward, terminated, truncated, info = env.step(action)
        rows.append(
            {
                "event": f"step_{step}",
                "action": ACTION_NAMES[action],
                "obs": int(obs),
                "reward": float(reward),
                "terminated": bool(terminated),
                "truncated": bool(truncated),
                "labels": sorted(info["labels"]),
                "constraint_step": info["constraint"]["step"],
                "constraint_episode": info["constraint"].get("episode", {}),
                "metrics_episode": info.get("metrics", {}).get("episode", {}),
            }
        )
        if terminated or truncated:
            break
    env.close()
    return rows

unsafe_rows = run_script(seed=1, actions=[2, 2, 2, 2])
goal_rows = run_script(seed=4, actions=[2] * 8 + [1] * 8)

print("Unsafe script final row:")
pprint(unsafe_rows[-1])
print("\nGoal script final row:")
pprint(goal_rows[-1])

assert unsafe_rows[-1]["constraint_step"]["violation"] == 1.0
assert unsafe_rows[-1]["constraint_episode"]["satisfied"] == 0.0
assert goal_rows[-1]["reward"] == 1.0
assert goal_rows[-1]["constraint_episode"]["satisfied"] == 1.0

## Algorithm registry

Each entry uses the same environment factory and seed pattern. The configuration is intentionally small and explicit so it is easy to compare what changes between algorithms.

In [ ]:
ALGORITHMS = {
    "q_learning": {
        "class": QL,
        "kwargs": {},
        "mechanism": "Task Q-table only; costs are logged but not penalized in the update target.",
    },
    "q_learning_lambda": {
        "class": QL_Lambda,
        "kwargs": {"cost_lambda": 1.0},
        "mechanism": "Subtracts cost_lambda * cost from the reward target.",
    },
    "lcrl": {
        "class": LCRL,
        "kwargs": {"r_min": -1.0},
        "mechanism": "Maps violations to an absorbing-style value based on r_min / (1 - gamma).",
    },
    "sem": {
        "class": SEM,
        "kwargs": {"r_min": -1.0, "cost_coef": 1.0},
        "mechanism": "Learns task Q plus auxiliary D and C safety tables that alter action selection.",
    },
    "recreg": {
        "class": RECREG,
        "kwargs": {
            "mode": "model_free",
            "model_checking": "none",
            "horizon": 3,
            "safety_prob": 0.2,
            "cost_coef": 2.0,
        },
        "mechanism": "Learns task Q, backup B, risk S, and can report override_rate when actions are replaced.",
    },
}

display(Markdown(markdown_table(
    [
        {"algorithm": name, "mechanism": spec["mechanism"]}
        for name, spec in ALGORITHMS.items()
    ],
    ["algorithm", "mechanism"],
)))

## Tiny smoke training runs

These calls intentionally use `num_frames=20`, `eval_freq=10`, `log_freq=10`, and `num_eval_episodes=1`. Treat the printed tables as wiring checks and diagnostics, not as claims about which method is best.

In [ ]:
def train_one(name, spec, seed=0):
    env = make_train_env()
    eval_env = make_eval_env()
    algo = spec["class"](
        env,
        eval_env=eval_env,
        seed=seed,
        device="cpu",
        verbose=0,
        exploration="epsilon_greedy",
        initial_epsilon=1.0,
        final_epsilon=0.2,
        epsilon_decay_frames=20,
        **spec["kwargs"],
    )
    algo.train(
        num_frames=20,
        eval_freq=10,
        log_freq=10,
        num_eval_episodes=1,
        stats_window_size=5,
    )
    env.close()
    eval_env.close()
    return algo

trained_algorithms = {}
for offset, (name, spec) in enumerate(ALGORITHMS.items()):
    print(f"\n=== {name} ===")
    trained_algorithms[name] = train_one(name, spec, seed=offset)

print("\n" + smoke_run_note())

## Inspect learned state

Even after a tiny run, the shape of each learned object shows what the method tracks. `QL`, `QL_Lambda`, and `LCRL` have a task `Q` table; `SEM` adds `D` and `C`; `RECREG` adds backup `B` and risk `S`.

In [ ]:
summary_rows = [
    algorithm_state_summary(name, algo)
    for name, algo in trained_algorithms.items()
]

display(Markdown(markdown_table(
    summary_rows,
    ["algorithm", "Q shape", "max Q", "min Q", "D shape", "C shape", "B shape", "S shape"],
)))

assert trained_algorithms["q_learning"].Q.shape == (81, 5)
assert trained_algorithms["q_learning_lambda"].cost_lambda == 1.0
assert hasattr(trained_algorithms["sem"], "D") and hasattr(trained_algorithms["sem"], "C")
assert hasattr(trained_algorithms["recreg"], "B") and hasattr(trained_algorithms["recreg"], "S")

## Reading the tradeoff

- `q_learning` is the task-only reference point. It is useful precisely because it does not encode a safety preference in its target.
- `q_learning_lambda` is the simplest penalty baseline: unsafe labels reduce the scalar return.
- `lcrl` treats violations more like entering a bad absorbing outcome than paying a small cost.
- `sem` keeps task and safety estimates separate, then combines them during action selection.
- `recreg` is intervention-oriented: it estimates risk, learns a backup policy, and can log `override_rate` when it replaces risky task actions.

For real comparisons, increase frames, run multiple seeds, record confidence intervals, and keep this notebook as a short correctness and interpretation check.